# CSE465 ColorBench — Faithful ACE (Agentic Context Engineering) Pipeline

**Architecture:** Agentic Context Engineering (ACE Framework — ICLR 2026, Stanford/SambaNova/Berkeley)
- **Roles:** Generator $\rightarrow$ Reflector $\rightarrow$ Curator $\rightarrow$ Persistent Playbook $\rightarrow$ Qwen2.5-VL-7B Solver.
- **Target Tasks:** Color Mimicry & Color Illusion on ColorBench (UMD, 2025).
- **Fair Evaluation:** Baseline and ACE are evaluated on the **exact same held-out evaluation instances** using fixed-seed splits.
- **Hardware Target:** Google Colab Free-Tier (Tesla T4 GPU, 15GB VRAM) in 4-bit NF4 precision.


## 1. Verify GPU Setup


In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"[GPU Ready] {gpu_name} ({vram:.1f} GB VRAM)")
else:
    print("WARNING: No GPU detected. Change Runtime type to GPU (T4) in Colab menu.")


## 2. Install Dependencies


In [ ]:
!pip install -q transformers bitsandbytes accelerate qwen-vl-utils datasets "pillow<11.0.0"


## 3. Clone / Pull Repository (`tanzim-ace` Branch)


In [ ]:
import os

# Update with your repository URL
REPO_URL = "https://github.com/tanzim12911/cse465-project.git"

%cd /content
if not os.path.exists("cse465-project"):
    !git clone -b tanzim-ace {REPO_URL}
else:
    %cd cse465-project
    !git checkout tanzim-ace
    !git pull origin tanzim-ace

%cd /content/cse465-project
!git status


## 4. Run Unit Tests (Verify ACE Components)


In [ ]:
!python -m unittest discover -s tests -p "test_*.py"


## 5. Select Model Configuration

Choose a model for your experiments on Colab T4 GPU:
- `"7b"` → `Qwen/Qwen2.5-VL-7B-Instruct` **(recommended)** — 4-bit NF4, ~7 GB VRAM peak. Better reasoning quality for the Generator/Reflector meta-cognitive tasks in ACE.
- `"3b"` → `Qwen/Qwen2.5-VL-3B-Instruct` — faster but weaker instruction following; higher JSON parse failure rate hurts playbook quality. Use only if hitting OOM on 7B.

**Memory note:** 7B NF4 uses ~7 GB peak on T4 (15 GB). The `min_pixels`/`max_pixels` caps in `config.py` prevent vision token OOM.


In [ ]:
# Select model: "7b" or "3b"
MODEL_ID = "7b"
print(f"Selected Model: {MODEL_ID}")


## 6. Primary Experiment 1: Color Mimicry (Baseline vs. ACE)

Runs both Baseline and ACE on the exact same held-out split (seed=42):
- **Phase 1 (Adaptation):** 15 samples (evolves persistent playbook starting from empty `{}`).
- **Phase 2 (Held-Out Evaluation):** 15 unseen samples (evaluates frozen learned playbook vs. zero-shot baseline).

> **Note:** 15 adaptation steps (up from 10) gives bad bullets enough signal to be suppressed before held-out evaluation.


In [ ]:
!python run_pipeline.py --task "Color Mimicry" --model_id {MODEL_ID} --mode both --num_adaptation 15 --num_eval 15 --seed 42


## 7. Primary Experiment 2: Color Illusion (Baseline vs. ACE)

Runs both Baseline and ACE on the exact same held-out split (seed=42):
- **Phase 1 (Adaptation):** 15 samples (evolves persistent playbook starting from empty `{}`).
- **Phase 2 (Held-Out Evaluation):** 15 unseen samples (evaluates frozen learned playbook vs. zero-shot baseline).

> **Note:** 15 adaptation steps (up from 10) gives bad bullets enough signal to be suppressed before held-out evaluation.


In [ ]:
!python run_pipeline.py --task "Color Illusion" --model_id {MODEL_ID} --mode both --num_adaptation 15 --num_eval 15 --seed 42


## 8. Comparative Evaluation Report

Summarizes held-out baseline accuracy vs. held-out ACE accuracy and delta improvements.


In [ ]:
!python eval_results.py --dir ./results


## 9. Inspect What ACE Learned (Evolved Playbooks)


In [ ]:
import glob
from IPython.display import display, Markdown

# Search recursively so both flat (results/playbook_*.md) and
# run-subfolder (results/run-N/playbook_*.md) layouts are covered.
playbook_md_files = glob.glob("results/**/playbook_*.md", recursive=True)
playbook_md_files += glob.glob("results/playbook_*.md")
playbook_md_files = list(dict.fromkeys(playbook_md_files))  # deduplicate

if not playbook_md_files:
    print("No playbook markdown files found under results/.")
else:
    for md_path in playbook_md_files:
        print(f"\n{'='*70}\nInspecting Playbook: {md_path}\n{'='*70}")
        with open(md_path, "r", encoding="utf-8") as f:
            display(Markdown(f.read()))
